# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/upen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/upen/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '1fea85'. Skipping!
Property 'summary' already exists in node 'a49817'. Skipping!
Property 'summary' already exists in node 'ccdc0e'. Skipping!
Property 'summary' already exists in node '86e399'. Skipping!
Property 'summary' already exists in node '8fcbe0'. Skipping!
Property 'summary' already exists in node '710d3d'. Skipping!
Property 'summary' already exists in node '6c1ab1'. Skipping!
Property 'summary' already exists in node 'c8dd24'. Skipping!
Property 'summary' already exists in node 'b8be31'. Skipping!
Property 'summary' already exists in node '2f8a42'. Skipping!
Property 'summary' already exists in node 'c75605'. Skipping!
Property 'summary' already exists in node '902580'. Skipping!
Property 'summary' already exists in node '41032d'. Skipping!
Property 'summary' already exists in node '704f18'. Skipping!
Property 'summary' already exists in node 'fd5ec7'. Skipping!
Property 'summary' already exists in node '6169c8'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'ccdc0e'. Skipping!
Property 'summary_embedding' already exists in node '86e399'. Skipping!
Property 'summary_embedding' already exists in node '8fcbe0'. Skipping!
Property 'summary_embedding' already exists in node '6c1ab1'. Skipping!
Property 'summary_embedding' already exists in node 'a49817'. Skipping!
Property 'summary_embedding' already exists in node '1fea85'. Skipping!
Property 'summary_embedding' already exists in node '710d3d'. Skipping!
Property 'summary_embedding' already exists in node 'b8be31'. Skipping!
Property 'summary_embedding' already exists in node 'c75605'. Skipping!
Property 'summary_embedding' already exists in node '2f8a42'. Skipping!
Property 'summary_embedding' already exists in node 'c8dd24'. Skipping!
Property 'summary_embedding' already exists in node '902580'. Skipping!
Property 'summary_embedding' already exists in node '41032d'. Skipping!
Property 'summary_embedding' already exists in node '6169c8'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 710)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 710)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

Answer :
-   "Single-Hop": The name implies that the answer requires only one "hop" or one piece of retrieved information. The system doesn't need to combine information from multiple sources."Specific": The questions are very focused and target concrete details found within a single document chunk. "0.50" :50% are simple, single-document questions.
-   "Multi-Hop": This indicates that answering the question requires at least two "hops"—retrieving and connecting information from different parts of your knowledge base. "Specific": Like the single-hop version, the final answer is still a concrete, specific piece of information. "0.25" : 25% are complex, multi-document questions requiring a specific answer.
-   "Multi-Hop": It still requires synthesizing information from various sources. "Abstract": The question prompts a higher-level understanding, comparison, or summarization rather than a single, discrete fact. "0.25" : 25% are complex, multi-document questions requiring an abstract or summary answer.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What do Handa et al. contribute to the underst...,[Introduction ChatGPT launched in November 202...,Handa et al. report statistics on chatbot conv...,single_hop_specifc_query_synthesizer
1,How does ChatGPT usage differ between work-rel...,[Table 1: ChatGPT daily message counts (millio...,Table 1 indicates that while total daily messa...,single_hop_specifc_query_synthesizer
2,ChatGPT use how in jobs?,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 shows variat...,single_hop_specifc_query_synthesizer
3,What significance does November 2022 hold in t...,[Conclusion This paper studies the rapid growt...,"ChatGPT was launched in November 2022, marking...",single_hop_specifc_query_synthesizer
4,"How do the increasing non-work messages, which...",[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"The data shows that in June 2024, non-work mes...",multi_hop_abstract_query_synthesizer
5,How do user behavior and usage trends over tim...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The context shows that ChatGPT's usage has gro...,multi_hop_abstract_query_synthesizer
6,how does LLM AI like ChatGPT impact society ec...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, based on Large Language Models (LLMs)...",multi_hop_abstract_query_synthesizer
7,how Claude use ChatGPT for work and non work a...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context shows that ChatGPT, launched in No...",multi_hop_specific_query_synthesizer
8,How does the development and adoption of Claud...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The provided context primarily discusses ChatG...,multi_hop_specific_query_synthesizer
9,How does the development and adoption of AI mo...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context indicates that ChatGPT, based on l...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a716f8'. Skipping!
Property 'summary' already exists in node 'c0aea4'. Skipping!
Property 'summary' already exists in node '5a57cb'. Skipping!
Property 'summary' already exists in node 'aa627f'. Skipping!
Property 'summary' already exists in node '9e6277'. Skipping!
Property 'summary' already exists in node '709fb2'. Skipping!
Property 'summary' already exists in node 'ecef95'. Skipping!
Property 'summary' already exists in node 'fa9a84'. Skipping!
Property 'summary' already exists in node '774422'. Skipping!
Property 'summary' already exists in node '02dbab'. Skipping!
Property 'summary' already exists in node 'e37727'. Skipping!
Property 'summary' already exists in node '653465'. Skipping!
Property 'summary' already exists in node 'b40cac'. Skipping!
Property 'summary' already exists in node 'f60009'. Skipping!
Property 'summary' already exists in node '959159'. Skipping!
Property 'summary' already exists in node '674240'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '774422'. Skipping!
Property 'summary_embedding' already exists in node '5a57cb'. Skipping!
Property 'summary_embedding' already exists in node 'a716f8'. Skipping!
Property 'summary_embedding' already exists in node 'fa9a84'. Skipping!
Property 'summary_embedding' already exists in node 'aa627f'. Skipping!
Property 'summary_embedding' already exists in node 'ecef95'. Skipping!
Property 'summary_embedding' already exists in node 'e37727'. Skipping!
Property 'summary_embedding' already exists in node 'c0aea4'. Skipping!
Property 'summary_embedding' already exists in node '9e6277'. Skipping!
Property 'summary_embedding' already exists in node '709fb2'. Skipping!
Property 'summary_embedding' already exists in node '02dbab'. Skipping!
Property 'summary_embedding' already exists in node '653465'. Skipping!
Property 'summary_embedding' already exists in node 'b40cac'. Skipping!
Property 'summary_embedding' already exists in node '674240'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Who are Wiggers?,[Introduction ChatGPT launched in November 202...,Wiggers (2025) reports estimates that in April...,single_hop_specifc_query_synthesizer
1,How does Claude compare to ChatGPT in terms of...,[Table 1: ChatGPT daily message counts (millio...,The context provides data on ChatGPT's daily m...,single_hop_specifc_query_synthesizer
2,What does SOC stand for in the context of Chat...,[Variation by Occupation Figure 23 presents va...,The context does not specify what SOC stands f...,single_hop_specifc_query_synthesizer
3,What does Writing mean in ChatGPT use?,[Conclusion This paper studies the rapid growt...,"Writing is by far the most common work use, ac...",single_hop_specifc_query_synthesizer
4,How do privacy considerations in data reportin...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that due to privacy-pres...,multi_hop_abstract_query_synthesizer
5,"H0w does the growht and adopti0n of ChatGPT, a...",[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth and adoption of ChatGPT, a ke...",multi_hop_abstract_query_synthesizer
6,Considering the variation in ChatGPT usage by ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that due to privacy-pres...,multi_hop_abstract_query_synthesizer
7,Wha is the varation in ChatGPT usag by occpati...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,Variation by Occupation Figure 23 shows that u...,multi_hop_abstract_query_synthesizer
8,US ChatGPT usage like in the US how much is no...,[<1-hop>\n\nConclusion This paper studies the ...,"The context shows that in the US, non-work mes...",multi_hop_specific_query_synthesizer
9,Based on the rapid growth of ChatGPT usage in ...,[<1-hop>\n\nConclusion This paper studies the ...,"The context indicates that by July 2025, over ...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE810122025"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the context provided from the document titled "How People Use ChatGPT," people are using AI, particularly generative AI like ChatGPT, for a variety of tasks including:\n\n- Performing workplace tasks, either augmenting or automating human labor.\n- Producing writing, software code, spreadsheets, and other digital products.\n- Seeking information and advice.\n- Using AI as co-workers that produce output or as co-pilots that give advice and improve productivity.\n- Engaging in activities related to self-expression such as relationships, personal reflection, games, and role play (though these represent smaller percentages of usage).\n- Using AI for economic tasks and labor market implications, indicating an occupational impact.\n\nOverall, generative AI is highly flexible and used for both work-related tasks and personal purposes, distinguishing it from traditional technologies like web search engines.\n\nTherefore, people today use AI to augment or automate work, generate vario

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates if the answer correctly addresses the question. It measures, "Does the answer directly respond to the question?","Is the answer factually correct?","Is the answer complete?"
- `labeled_helpfulness_evaluator`: Evaluates how helpful the response is compared to a reference answer. It measures, "How well the RAG answer matches the expected/reference  answer".
- `dopeness_evaluator`: Evaulates the coolness or engagment level of responses using "criteria". It measures, "Is response engaging and interesting?","Is it generic and boring?","Does it have personality and flair?".

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'brief-plant-18' at:
https://smith.langchain.com/o/87477bb6-977f-48b7-8e7f-df659e82c25d/datasets/ee38e568-dc40-4583-86f5-ae39f9bc24db/compare?selectedSessions=a7104243-9ccd-41e4-964c-fff660645bfb




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does OpenAI's development and widespread a...,OpenAI's development and widespread adoption o...,None,The context shows that OpenAI launched ChatGPT...,1,1,0,8.331451,b9692c90-003a-4445-bebd-080a0b006a79,2e0abe7c-808d-400a-971b-945fc4c3ae38
1,Based on the data presented about ChatGPT's us...,I don't know.,None,The context indicates that ChatGPT's usage has...,0,0,0,0.719829,58abd4a3-0051-4f87-a1c2-0a0bb12eaf16,09df6234-2ea2-4fcb-959a-41fa97b632b7
2,Based on the rapid growth of ChatGPT usage in ...,"Based on the provided context, the increasing ...",None,"The context indicates that by July 2025, over ...",1,1,0,2.682743,40597422-da2f-46e9-9c06-2b5a7ad42e6f,40c321f5-85c8-43cc-9fc8-8bcf3c443176
3,US ChatGPT usage like in the US how much is no...,Based on the provided context:\n\n- In June 20...,None,"The context shows that in the US, non-work mes...",1,0,0,3.892264,0c391f3a-d3bf-47e4-8e5b-1300a51a50de,1421f93a-d90f-4e21-830f-c1b60d659410
4,Wha is the varation in ChatGPT usag by occpati...,The variation in ChatGPT usage by occupation s...,None,Variation by Occupation Figure 23 shows that u...,1,0,0,3.878799,9876d5af-2c73-4f47-9d75-5e266053019b,6a74a5a2-6d85-4561-9525-fd6458299bfa
5,Considering the variation in ChatGPT usage by ...,Privacy concerns lead to aggregation of data i...,None,The context indicates that due to privacy-pres...,1,1,0,4.878473,2eed31a7-ea46-4a28-9296-6976c183a9ca,4fe7688b-c8c1-49ee-aba1-b3a1ea30db1d
6,"H0w does the growht and adopti0n of ChatGPT, a...","Based on the provided context, the rapid growt...",None,"The rapid growth and adoption of ChatGPT, a ke...",1,1,0,4.722778,f2790de7-1d3a-485e-8b94-bbc8a2b97c96,8c9ba750-2cb8-4e8a-9e1c-6370b1abe100
7,How do privacy considerations in data reportin...,Privacy considerations in data reporting lead ...,None,The context indicates that due to privacy-pres...,1,0,0,2.110472,044a0411-04a9-47a1-86ef-fb66ffe97369,1c71c305-76b2-45e2-93ef-9e71393b289c
8,What does Writing mean in ChatGPT use?,"Based on the context provided, ""Writing"" in Ch...",None,"Writing is by far the most common work use, ac...",1,1,0,3.111129,adf8116e-3a01-46fc-9f31-c45dc111f1a7,380580fa-1b3c-4bd3-ad7c-e8c692d7263a
9,What does SOC stand for in the context of Chat...,I don't know.,None,The context does not specify what SOC stands f...,0,0,0,0.864665,015140a9-8d56-4548-a130-9c252e1b931c,830eed8c-3836-43cb-92a8-2dcdcad0a040


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

Answer: Larger chunks provide more complete context for the LLM to work with. Less likely to miss important context that was split across chuk boundaries. The synthetic data includes multi hop questions that requires information from multi parts of the document.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

Answer : It provides higher quality embeddings as it uses 3072 dimensions. With better embeddings, can achieve better semantic matching which means can capture subtle relationships between concepts. This inturn improves ranking of documents. For multi hop questions, it requires better embeddings. The cons is, it slower, more expensie and higher computational requirments.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, hold onto your hats — here’s the lowdown on how folks are turning AI into cold, hard cash, straight from the dopest insights in the context:\n\nPeople are not just using AI like ChatGPT to crank out tasks robotically. Nah, they’re tapping into it as a slick advisor and research sidekick, leveling up their decision-making game. Especially in knowledge-heavy gigs, AI boosts productivity by sharpening the quality of decisions — that’s the secret sauce turning brainpower into bank.\n\nThink of ChatGPT as your turbocharged wingman in the workplace, supporting complex decision processes, giving advice, and helping you finesse your craft. This is where the money magic happens: by augmenting human smarts instead of just automating grunt work, AI makes workers *more* effective, which translates to more opportunities to earn and create value.\n\nSo the hustle? Use AI as a decision-support ninja and research guru to step up your professional game, wielding better info and sharper insigh

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'slight-crown-78' at:
https://smith.langchain.com/o/87477bb6-977f-48b7-8e7f-df659e82c25d/datasets/ee38e568-dc40-4583-86f5-ae39f9bc24db/compare?selectedSessions=25194df6-fef0-4326-8c56-576de0e04034




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does OpenAI's development and widespread a...,"Oh, buckle up, because OpenAI’s ChatGPT story ...",None,The context shows that OpenAI launched ChatGPT...,1,1,1,5.767229,b9692c90-003a-4445-bebd-080a0b006a79,45eabb13-1d54-4527-87af-5fac5bacfa76
1,Based on the data presented about ChatGPT's us...,"Yo, diving into this cosmic swirl of ChatGPT’s...",None,The context indicates that ChatGPT's usage has...,1,0,1,4.862847,58abd4a3-0051-4f87-a1c2-0a0bb12eaf16,2fb51123-a2f1-42d1-9d15-f049b33c31f4
2,Based on the rapid growth of ChatGPT usage in ...,"Alright, here’s the high-voltage juice on this...",None,"The context indicates that by July 2025, over ...",1,1,1,5.081415,40597422-da2f-46e9-9c06-2b5a7ad42e6f,ea9e0ec0-73af-407c-800a-ed04944058d6
3,US ChatGPT usage like in the US how much is no...,"Yo, here’s the lowdown straight from the cryst...",None,"The context shows that in the US, non-work mes...",0,0,1,3.450995,0c391f3a-d3bf-47e4-8e5b-1300a51a50de,e681b81e-7ab5-442f-9dec-f209b9a5cebb
4,Wha is the varation in ChatGPT usag by occpati...,"Yo, let’s unpack this ChatGPT usage saga by oc...",None,Variation by Occupation Figure 23 shows that u...,1,0,1,10.348350,9876d5af-2c73-4f47-9d75-5e266053019b,8cc58254-c36a-409f-9425-729ed419d94f
5,Considering the variation in ChatGPT usage by ...,"Alright, buckle up: privacy is the unsung hero...",None,The context indicates that due to privacy-pres...,1,1,1,5.705936,2eed31a7-ea46-4a28-9296-6976c183a9ca,87ad13ac-d997-4da4-b34d-4910d9782e07
6,"H0w does the growht and adopti0n of ChatGPT, a...","Yo, here’s the straight-up nitty-gritty: As of...",None,"The rapid growth and adoption of ChatGPT, a ke...",1,1,1,3.967848,f2790de7-1d3a-485e-8b94-bbc8a2b97c96,a366e5f3-6ed1-4218-88ea-6ed572559948
7,How do privacy considerations in data reportin...,"Alright, buckle up—here's the scoop with a tur...",None,The context indicates that due to privacy-pres...,1,1,1,4.212548,044a0411-04a9-47a1-86ef-fb66ffe97369,d4a0922f-7f18-4a78-99c2-a1d8289cb6ea
8,What does Writing mean in ChatGPT use?,"Alright, buckle up — Writing in the ChatGPT un...",None,"Writing is by far the most common work use, ac...",1,1,1,3.840713,adf8116e-3a01-46fc-9f31-c45dc111f1a7,de817f52-49c6-4c3b-a58c-3ecfbad75bd7
9,What does SOC stand for in the context of Chat...,"Oh, you’re diving deep into the gears behind C...",None,The context does not specify what SOC stands f...,0,0,1,3.981829,015140a9-8d56-4548-a130-9c252e1b931c,5d99ddb7-4cf3-43a3-a757-51f4ca618323


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

Answer : 
- In below snapshot, brief-plant-18 refers to the original version(small embeddings, chunk size: 500) and slight-crown-78 referes to improved version (large embeddings, chuk size :1000, dopness prompt)
- Observations
    -   The original version have failed 4 scearnios. The one contributing factor is no relevant context.
    -   The improved version have got high scores across many questions and inline with reference context. This could be due to chunk size and also better embeddings.
    -   The imporved version took longer time for processing. This is anticipated due to large embeddings.

    

![Evaluation Comparison Results](Eval_Compare_Results.png)


![Evaluation Comparison Results 2](Eval_Compare_Results2.png)
